# Asynchronous Tick-to-Alpha & Execution Engine
Instead of going through Kaggle or LOBSTER for data, I've decided to generate it myself.

In [9]:
import numpy as np
import pandas as pd

def generate_asynchronous_market_data( n_quotes: int = 50_000, n_trades: int = 15_000, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Simulates irregular Level-1 BBO quotes and asynchronous trade prints
    with embedded Order Book Imbalance (OBI) alpha and realistic feed corruptions.
    """
    rng = np.random.default_rng(seed)
    start_time = pd.Timestamp("2026-11-12 09:30:00.000")
    quote_dt_ms = rng.exponential(scale=15.0, size=n_quotes)  # avg 15ms between quotes
    quote_ts = start_time + pd.to_timedelta(np.cumsum(quote_dt_ms), unit="ms")

    trade_dt_ms = rng.exponential(scale=50.0, size=n_trades)  # avg 50ms between trades
    trade_ts = start_time + pd.to_timedelta(np.cumsum(trade_dt_ms), unit="ms")
    bid_sz = rng.integers(50, 1000, size=n_quotes).astype(float)
    ask_sz = rng.integers(50, 1000, size=n_quotes).astype(float)
    obi = (bid_sz - ask_sz) / (bid_sz + ask_sz)

    alpha_signal = 0.0015 * pd.Series(obi).shift(1).fillna(0.0).to_numpy()
    noise = rng.normal(0.0, 0.008, size=n_quotes)
    mid_px = 100.0 + np.cumsum(alpha_signal + noise)

    half_spread = rng.choice([0.005, 0.010, 0.015], size=n_quotes, p=[0.6, 0.3, 0.1])
    bid_px = np.round(mid_px - half_spread, 2)
    ask_px = np.round(mid_px + half_spread, 2)

    quotes = pd.DataFrame({
        "timestamp": quote_ts,
        "bid_px": bid_px,
        "ask_px": ask_px,
        "bid_sz": bid_sz,
        "ask_sz": ask_sz,
    })

    prev_idx = np.searchsorted(quote_ts, trade_ts, side="right") - 1
    prev_idx = np.clip(prev_idx, 0, n_quotes - 1)

    # aggressor buys hit ask (+1), sells hit bid (-1), influenced by prevailing OBI
    buy_prob = 0.5 + 0.25 * obi[prev_idx]
    is_buy = rng.random(n_trades) < buy_prob
    trade_px = np.where(is_buy, ask_px[prev_idx], bid_px[prev_idx])
    trade_sz = rng.integers(10, 500, size=n_trades).astype(float)

    trades = pd.DataFrame({
        "timestamp": trade_ts,
        "trade_px": trade_px,
        "trade_sz": trade_sz,
    })

    # inject data hygiene traps so i can cleane them later
    corrupt_idx = rng.choice(n_quotes, size=50, replace=False)
    quotes.loc[corrupt_idx, "bid_px"] = quotes.loc[corrupt_idx, "ask_px"] + 0.02

    nan_idx = rng.choice(n_quotes, size=30, replace=False)
    quotes.loc[nan_idx, "ask_sz"] = 0.0

    shuffle_slice = quotes.iloc[100:110].sample(frac=1.0, random_state=seed)
    quotes.iloc[100:110] = shuffle_slice.to_numpy()

    quotes.loc[10, "timestamp"] = trades.loc[3, "timestamp"]

    return quotes, trades

quotes_raw, trades_raw = generate_asynchronous_market_data()
print("Quotes shape:", quotes_raw.shape)
print("Trades shape:", trades_raw.shape)
display(quotes_raw.head(5))
display(trades_raw.head(5))

Quotes shape: (50000, 5)
Trades shape: (15000, 3)


,timestamp,bid_px,ask_px,bid_sz,ask_sz
0,2026-11-12 09:30:00.036063129,99.99,100.01,976.0,172.0
1,2026-11-12 09:30:00.071105974,100.00,100.01,688.0,440.0
2,2026-11-12 09:30:00.106877389,100.02,100.03,169.0,541.0
3,2026-11-12 09:30:00.111074303,100.02,100.03,434.0,162.0
4,2026-11-12 09:30:00.112370864,100.03,100.04,779.0,356.0


,timestamp,trade_px,trade_sz
0,2026-11-12 09:30:00.075534362,100.01,112.0
1,2026-11-12 09:30:00.091019219,100.01,355.0
2,2026-11-12 09:30:00.091676734,100.00,79.0
3,2026-11-12 09:30:00.097851824,100.00,364.0
4,2026-11-12 09:30:00.155951199,100.04,248.0
